# TP3 - Análisis de las tablas

La idea es mirar primero qué información tenemos en cada tabla y después ver cómo relacionarlas sin usar datos del futuro. La unidad principal va a ser la solicitud: cuando terminemos de preparar los datos tiene que quedar una fila por solicitud.

In [1]:
import pandas as pd

solicitudes_train = pd.read_csv("data/solicitudes_train.csv")
comportamiento_bancario = pd.read_csv("data/comportamiento_bancario.csv")

## Solicitudes de préstamos

Cada fila representa una solicitud. `ID_Solicitud` identifica la solicitud y `ID_Usuario` nos va a permitir relacionarla con el historial bancario.

In [2]:
solicitudes_train.head()

,ID_Solicitud,ID_Usuario,Monto_Solicitado,Tasa_Interes,Ingreso_Mensual,Motivo_Prestamo,Fecha_Solicitud,Estado_Final
0,1,12698,7900,0.2333,4461,Consumo,2025-04,1
1,2,20744,17400,0.2507,3889,Vivienda,2025-12,1
2,3,6830,8600,0.2368,4035,Vehiculo,2025-06,0
3,4,5722,13300,0.2826,4018,Vehiculo,2025-07,1
4,5,24719,6600,0.2314,3357,Otros,2025-01,1


In [3]:
solicitudes_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 32416 entries, 0 to 32415
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ID_Solicitud      32416 non-null  int64  
 1   ID_Usuario        32416 non-null  int64  
 2   Monto_Solicitado  32416 non-null  int64  
 3   Tasa_Interes      32416 non-null  float64
 4   Ingreso_Mensual   32416 non-null  int64  
 5   Motivo_Prestamo   32416 non-null  str    
 6   Fecha_Solicitud   32416 non-null  str    
 7   Estado_Final      32416 non-null  int64  
dtypes: float64(1), int64(5), str(2)
memory usage: 2.4 MB


Primero revisamos la cantidad de usuarios, si hay duplicados o datos faltantes y qué período cubren las solicitudes. También pasamos la fecha al tipo correcto para poder compararla más adelante con los meses del historial bancario.

In [4]:
solicitudes_train["Fecha_Solicitud"] = pd.to_datetime(
    solicitudes_train["Fecha_Solicitud"]
)

print("Usuarios distintos:", solicitudes_train["ID_Usuario"].nunique())
print("Filas duplicadas:", solicitudes_train.duplicated().sum())
print("IDs de solicitud repetidos:", solicitudes_train["ID_Solicitud"].duplicated().sum())
print("\nValores faltantes:")
print(solicitudes_train.isnull().sum())
print("\nPeríodo:", solicitudes_train["Fecha_Solicitud"].min().date(),
      "a", solicitudes_train["Fecha_Solicitud"].max().date())
print("Máximo de solicitudes de un usuario:",
      solicitudes_train.groupby("ID_Usuario").size().max())

Usuarios distintos: 18548
Filas duplicadas: 0
IDs de solicitud repetidos: 0

Valores faltantes:
ID_Solicitud        0
ID_Usuario          0
Monto_Solicitado    0
Tasa_Interes        0
Ingreso_Mensual     0
Motivo_Prestamo     0
Fecha_Solicitud     0
Estado_Final        0
dtype: int64

Período: 2025-01-01 a 2025-12-01
Máximo de solicitudes de un usuario: 8


### Variables numéricas

Esto nos da una idea general de los montos solicitados, los ingresos y las tasas de interés.

In [5]:
solicitudes_train[[
    "Monto_Solicitado",
    "Ingreso_Mensual",
    "Tasa_Interes"
]].describe()

,Monto_Solicitado,Ingreso_Mensual,Tasa_Interes
count,32416.000000,32416.000000,32416.000000
mean,14290.097483,4214.475938,0.259580
std,12995.295179,2498.904569,0.032333
min,500.000000,353.000000,0.147600
25%,5700.000000,2481.000000,0.237300
50%,10200.000000,3621.000000,0.258300
75%,18100.000000,5276.000000,0.280600
max,70000.000000,22000.000000,0.379900


### Relación entre el préstamo y el ingreso

El monto por sí solo no dice tanto. Podemos compararlo con el ingreso mensual de la persona. Un ratio más alto significa que el préstamo representa varios meses de ingreso y podría ser más difícil de pagar.

In [6]:
solicitudes_train["ratio_prestamo_ingreso"] = (
    solicitudes_train["Monto_Solicitado"]
    / solicitudes_train["Ingreso_Mensual"]
)

solicitudes_train[[
    "ID_Solicitud",
    "Monto_Solicitado",
    "Ingreso_Mensual",
    "ratio_prestamo_ingreso"
]].head()

,ID_Solicitud,Monto_Solicitado,Ingreso_Mensual,ratio_prestamo_ingreso
0,1,7900,4461,1.770903
1,2,17400,3889,4.474158
2,3,8600,4035,2.131351
3,4,13300,4018,3.310105
4,5,6600,3357,1.966041


### Estado final

`Estado_Final` es lo que más adelante queremos predecir: 1 significa que el préstamo fue pagado y 0 que terminó en default. Miramos las proporciones para saber si las dos clases están equilibradas.

In [7]:
solicitudes_train["Estado_Final"].value_counts(normalize=True).sort_index() * 100

Estado_Final
0    15.951999
1    84.048001
Name: proportion, dtype: float64

### Motivo del préstamo

Podemos mirar qué porcentaje se pagó dentro de cada categoría. Las diferencias no son enormes, pero la categoría puede aportar información y después podemos comparar el modelo con y sin ella.

In [8]:
porcentaje_pago_por_motivo = (
    solicitudes_train.groupby("Motivo_Prestamo")["Estado_Final"]
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

porcentaje_pago_por_motivo

Motivo_Prestamo
Educacion    85.990712
Vivienda     85.939941
Salud        85.610264
Consumo      83.677754
Otros        82.909640
Vehiculo     82.612782
Negocio      80.626672
Name: Estado_Final, dtype: float64

### Tasa de interés

La tasa promedio cambia poco entre los préstamos pagados y los que terminaron en default. Puede que no sea una variable muy fuerte para predecir por sí sola, pero no conviene descartarla todavía. Además, después la vamos a necesitar para calcular la ganancia esperada.

In [9]:
solicitudes_train.groupby("Estado_Final")["Tasa_Interes"].mean().sort_index()

Estado_Final
0    0.267905
1    0.258000
Name: Tasa_Interes, dtype: float64

## Comportamiento bancario

En esta tabla cada fila representa un mes de un usuario. Por eso tiene muchas más filas que la tabla de solicitudes.

In [10]:
comportamiento_bancario.head()

,ID_Usuario,Mes_Referencia,Saldo_Promedio_Mensual,Dias_Atraso_Otras_Deudas
0,1,2024-01,12768.11,3
1,1,2024-02,15817.67,3
2,1,2024-03,13138.66,0
3,1,2024-04,19453.13,0
4,1,2024-05,14943.44,59


In [11]:
comportamiento_bancario.info()

<class 'pandas.DataFrame'>
RangeIndex: 661139 entries, 0 to 661138
Data columns (total 4 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   ID_Usuario                661139 non-null  int64  
 1   Mes_Referencia            661139 non-null  str    
 2   Saldo_Promedio_Mensual    661139 non-null  float64
 3   Dias_Atraso_Otras_Deudas  661139 non-null  int64  
dtypes: float64(1), int64(2), str(1)
memory usage: 24.6 MB


Pasamos `Mes_Referencia` a fecha y revisamos usuarios, duplicados, valores faltantes y cantidad de meses disponibles.

In [12]:
comportamiento_bancario["Mes_Referencia"] = pd.to_datetime(
    comportamiento_bancario["Mes_Referencia"]
)

print("Usuarios distintos:", comportamiento_bancario["ID_Usuario"].nunique())
print("Filas duplicadas:", comportamiento_bancario.duplicated().sum())
print("Duplicados de usuario y mes:",
      comportamiento_bancario.duplicated(
          subset=["ID_Usuario", "Mes_Referencia"]
      ).sum())
print("Valores faltantes:")
print(comportamiento_bancario.isnull().sum())
print("Período:", comportamiento_bancario["Mes_Referencia"].min().date(),
      "a", comportamiento_bancario["Mes_Referencia"].max().date())

meses_por_usuario = comportamiento_bancario.groupby(
    "ID_Usuario"
)["Mes_Referencia"].nunique()

print("\nMeses disponibles por usuario:")
display(meses_por_usuario.describe().round(2))

Usuarios distintos: 28993
Filas duplicadas: 0
Duplicados de usuario y mes: 0
Valores faltantes:
ID_Usuario                  0
Mes_Referencia              0
Saldo_Promedio_Mensual      0
Dias_Atraso_Otras_Deudas    0
dtype: int64
Período: 2024-01-01 a 2025-12-01

Meses disponibles por usuario:


count    28993.00
mean        22.80
std          1.07
min         17.00
25%         22.00
50%         23.00
75%         24.00
max         24.00
Name: Mes_Referencia, dtype: float64

### Saldos y atrasos

`Saldo_Promedio_Mensual` no es el sueldo de la persona: es el saldo promedio que mantuvo en la cuenta durante ese mes. Nos puede servir para ver su situación y si viene mejorando o empeorando. Los días de atraso muestran cómo se comportó con otras deudas.

In [13]:
comportamiento_bancario[[
    "Saldo_Promedio_Mensual",
    "Dias_Atraso_Otras_Deudas"
]].describe()

,Saldo_Promedio_Mensual,Dias_Atraso_Otras_Deudas
count,661139.000000,661139.000000
mean,5463.296614,8.950755
std,5473.861364,13.199638
min,3.300000,0.000000
25%,2061.955000,1.000000
50%,3974.280000,5.000000
75%,7082.925000,10.000000
max,165915.350000,90.000000


Como exploración podemos resumir el historial completo de cada usuario para ver qué características podrían servir: saldo promedio, saldo inicial, saldo final, cambio del saldo y atrasos. Este resumen todavía no se usa para entrenar porque incluye meses que podrían ser posteriores a una solicitud.

In [14]:
banco_ordenado = comportamiento_bancario.sort_values(
    ["ID_Usuario", "Mes_Referencia"]
)

resumen_exploratorio_banco = banco_ordenado.groupby("ID_Usuario").agg(
    meses_disponibles=("Mes_Referencia", "count"),
    saldo_promedio=("Saldo_Promedio_Mensual", "mean"),
    saldo_inicial=("Saldo_Promedio_Mensual", "first"),
    saldo_final=("Saldo_Promedio_Mensual", "last"),
    atraso_promedio=("Dias_Atraso_Otras_Deudas", "mean"),
    atraso_maximo=("Dias_Atraso_Otras_Deudas", "max")
)

resumen_exploratorio_banco["cambio_saldo"] = (
    resumen_exploratorio_banco["saldo_final"]
    - resumen_exploratorio_banco["saldo_inicial"]
)

resumen_exploratorio_banco.head()

,meses_disponibles,saldo_promedio,saldo_inicial,saldo_final,atraso_promedio,atraso_maximo,cambio_saldo
ID_Usuario,,,,,,,
1,23,20017.683913,12768.11,23343.81,8.304348,59,10575.70
2,21,3781.878571,3335.41,4576.35,16.619048,67,1240.94
3,22,4577.649545,4683.17,4133.35,1.090909,4,-549.82
4,24,4992.794583,3971.51,6122.52,1.958333,18,2151.01
5,24,2556.866667,3738.41,1753.87,27.000000,70,-1984.54


## Cómo relacionar las tablas

No podemos unir y dejar las tablas así nomás porque una solicitud se repetiría una vez por cada mes bancario del usuario. Primero hacemos esa unión para relacionar los registros, después sacamos los meses futuros y finalmente vamos a resumir el historial de cada solicitud.

### Paso 1: unir por usuario

En este punto es normal que aumente mucho la cantidad de filas. Cada solicitud queda repetida por todos los meses bancarios de su usuario.

In [15]:
historial_solicitudes = solicitudes_train.merge(
    comportamiento_bancario,
    on="ID_Usuario",
    how="left"
)

print("Filas de solicitudes:", len(solicitudes_train))
print("Filas después del merge:", len(historial_solicitudes))

Filas de solicitudes: 32416
Filas después del merge: 739137


### Paso 2: conservar solamente los meses anteriores

Para una solicitud solo podemos usar la información que ya existía en ese momento. Si usáramos meses posteriores estaríamos mirando el futuro. Como las fechas indican solamente el mes, también dejamos afuera el mismo mes de la solicitud.

In [16]:
historial_solicitudes = historial_solicitudes[
    historial_solicitudes["Mes_Referencia"]
    < historial_solicitudes["Fecha_Solicitud"]
].copy()

print("Filas después del filtro temporal:", len(historial_solicitudes))
print("Solicitudes distintas:", historial_solicitudes["ID_Solicitud"].nunique())

Filas después del filtro temporal: 538638
Solicitudes distintas: 32416


Todavía hay varias filas por solicitud, una por cada mes anterior. El próximo paso va a ser resumir esos meses por `ID_Solicitud` para volver a tener una sola fila por préstamo.